# 01 — Exploratory data analysis (train + val only)

**Test isolation:** this notebook never loads test images or test metadata. `eda.load_eda_frame` can only return train/val rows; the only thing shown about test is its per-class count. Run with `make eda` after `make data`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

from defect_detection.config import load_config
from defect_detection.data import eda

# nbconvert runs from notebooks/; resolve paths from the repo root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
config = load_config(Path(os.environ.get("DD_CONFIG", "configs/train.yaml")))
RAW, PROCESSED = config.data.raw_dir, config.data.processed_dir
FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Split sizes (test: counts only) and exclusions

In [ ]:
eda.split_counts(PROCESSED)

In [ ]:
report = json.loads((PROCESSED / "split_report.json").read_text(encoding="utf-8"))
dedupe = report["dedupe"]
pd.Series({k: v for k, v in dedupe.items() if not isinstance(v, (dict, list))})

In [ ]:
print("cluster size histogram:", dedupe["cluster_size_histogram"])
print("near-dup distance histogram:", dedupe["near_duplicate_distance_histogram"])
print("clusters with conflicting labels:", dedupe["clusters_with_conflicting_labels"])

## 2. Class balance (train + val)

In [ ]:
frame = eda.load_eda_frame(PROCESSED)
balance = pd.crosstab(frame["split"], frame["class_name"], margins=True)
balance["defect_rate"] = (balance["defective"] / balance["All"]).round(3)
balance

## 3. Image sizes and modes

In [ ]:
frame.groupby(["width", "height", "mode"]).size().rename("images").to_frame()

In [ ]:
frame["size_bytes"].describe().round(0)

## 4. Lighting-shortcut check

If global brightness or contrast alone separates the classes, the model could learn the capture conditions instead of the defect. `separability_auc` near 0.5 means no shortcut; near 1.0 means a strong one.

In [ ]:
stats = eda.image_stats(frame, RAW)
eda.lighting_shortcut(stats)

In [ ]:
fig = eda.feature_histograms(stats)
fig.savefig(FIGURES / "eda_brightness_contrast.png", dpi=120)

## 5. Sample images per class

In [ ]:
fig = eda.sample_grid(frame, RAW, per_class=8, seed=config.seed)
fig.savefig(FIGURES / "eda_samples.png", dpi=120)

## 6. Largest duplicate clusters (augmented copies / label-noise spot check)

Clusters are groups of exact or rotation/flip-aware near-duplicates. They show whether the raw data already contains augmented variants of the same part.

In [ ]:
fig = eda.cluster_grid(frame, RAW, max_clusters=6)
if fig is not None:
    fig.savefig(FIGURES / "eda_duplicate_clusters.png", dpi=120)

## 7. Findings

Written after reviewing the outputs above on the real dataset (see the V1 gate report).